In [ ]:
# ============================================================
# NLP TEXT PREPROCESSING
# Dataset: SMS Spam Collection
# ============================================================

# 1. Install contractions
!pip install contractions -q


# 2. Import libraries
import pandas as pd
import re
import nltk
import contractions
import spacy

from nltk.stem import PorterStemmer
from IPython.display import display


# ============================================================
# 3. DOWNLOAD NLTK RESOURCES
# ============================================================

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')


# ============================================================
# 4. LOAD SPACY ENGLISH MODEL
# ============================================================

nlp = spacy.load("en_core_web_sm")


# ============================================================
# 5. INITIALIZE STEMMER
# ============================================================

stemmer = PorterStemmer()


# ============================================================
# 6. READ SMS SPAM COLLECTION DATASET
# ============================================================

df = pd.read_csv(
    "spam.csv",
    encoding="latin-1"
)


# ============================================================
# 7. SELECT USEFUL COLUMNS
# ============================================================

df = df[['v1', 'v2']]
df.columns = ['label', 'message']


# ============================================================
# 8. REMOVE MISSING VALUES
# ============================================================

df.dropna(
    subset=['message'],
    inplace=True
)


# ============================================================
# 9. REMOVE DUPLICATE MESSAGES
# ============================================================

df.drop_duplicates(
    subset=['message'],
    inplace=True
)


# ============================================================
# 10. KEEP ORIGINAL MESSAGE
# ============================================================

df['original_message'] = df['message']


# ============================================================
# TEXT CLEANING
# ============================================================

# 11. Convert text to lowercase

df['message'] = df['message'].str.lower()


# 12. Expand contractions

df['message'] = df['message'].apply(
    contractions.fix
)


# 13. Remove punctuation and numbers

df['message'] = df['message'].apply(
    lambda x: re.sub(
        r'[^a-zA-Z\s]',
        '',
        x
    )
)


# 14. Remove extra whitespace

df['message'] = df['message'].apply(
    lambda x: re.sub(
        r'\s+',
        ' ',
        x
    ).strip()
)


# ============================================================
# TOKENIZATION
# ============================================================

df['tokens'] = df['message'].apply(
    nltk.word_tokenize
)


# ============================================================
# STOP-WORD REMOVAL
# ============================================================

stop_words = set(
    nltk.corpus.stopwords.words('english')
)

df['tokens'] = df['tokens'].apply(
    lambda x: [
        word
        for word in x
        if word not in stop_words
    ]
)


# ============================================================
# LEMMATIZATION
# ============================================================

def lemmatize_words(words):

    text = " ".join(words)

    doc = nlp(text)

    return [
        word.lemma_
        for word in doc
        if not word.is_space
    ]


df['lemmatized_tokens'] = df['tokens'].apply(
    lemmatize_words
)


# ============================================================
# STEMMING
# ============================================================

df['stemmed_tokens'] = df['tokens'].apply(
    lambda x: [
        stemmer.stem(word)
        for word in x
    ]
)


# ============================================================
# CONVERT TOKENS BACK TO TEXT
# ============================================================

df['lemmatized_text'] = df[
    'lemmatized_tokens'
].apply(
    lambda x: " ".join(x)
)


df['stemmed_text'] = df[
    'stemmed_tokens'
].apply(
    lambda x: " ".join(x)
)


# ============================================================
# FINAL TABLE — ONLY 5 ROWS
# ============================================================

print("\nORIGINAL DATA VS PREPROCESSED DATA\n")

final_data = df[
    [
        'label',
        'original_message',
        'message',
        'tokens',
        'lemmatized_text',
        'stemmed_text'
    ]
].head(5).copy()


# Rename columns

final_data.columns = [
    'Label',
    'Original SMS',
    'Cleaned SMS',
    'Tokens',
    'Lemmatized Text',
    'Stemmed Text'
]


# Display table

display(
    final_data.style
    .set_properties(
        **{
            'text-align': 'left',
            'white-space': 'pre-wrap'
        }
    )
    .set_table_styles([
        {
            'selector': 'th',
            'props': [
                ('background-color', '#2E7D32'),
                ('color', 'white'),
                ('font-weight', 'bold'),
                ('text-align', 'center')
            ]
        },
        {
            'selector': 'td',
            'props': [
                ('vertical-align', 'top')
            ]
        }
    ])
)


# ============================================================
# SAVE PREPROCESSED DATASET
# ============================================================

df.to_csv(
    "preprocessed_sms.csv",
    index=False
)

print("\nPreprocessing completed successfully!")
print("Saved as: preprocessed_sms.csv")


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!



ORIGINAL DATA VS PREPROCESSED DATA



,Label,Original SMS,Cleaned SMS,Tokens,Lemmatized Text,Stemmed Text
0,ham,"Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...",go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat,"['go', 'jurong', 'point', 'crazy', 'available', 'bugis', 'n', 'great', 'world', 'la', 'e', 'buffet', 'cine', 'got', 'amore', 'wat']",go jurong point crazy available bugis n great world la e buffet cine get amore wat,go jurong point crazi avail bugi n great world la e buffet cine got amor wat
1,ham,Ok lar... Joking wif u oni...,ok lar joking wif you oni,"['ok', 'lar', 'joking', 'wif', 'oni']",ok lar joking wif oni,ok lar joke wif oni
2,spam,Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's,free entry in a wkly comp to win fa cup final tkts st may text fa to to receive entry questionstd txt ratetcs apply overs,"['free', 'entry', 'wkly', 'comp', 'win', 'fa', 'cup', 'final', 'tkts', 'st', 'may', 'text', 'fa', 'receive', 'entry', 'questionstd', 'txt', 'ratetcs', 'apply', 'overs']",free entry wkly comp win fa cup final tkts st may text fa receive entry questionstd txt ratetcs apply over,free entri wkli comp win fa cup final tkt st may text fa receiv entri questionstd txt ratetc appli over
3,ham,U dun say so early hor... U c already then say...,you dun say so early hor you c already then say,"['dun', 'say', 'early', 'hor', 'c', 'already', 'say']",dun say early hor c already say,dun say earli hor c alreadi say
4,ham,"Nah I don't think he goes to usf, he lives around here though",nah i do not think he goes to usf he lives around here though,"['nah', 'think', 'goes', 'usf', 'lives', 'around', 'though']",nah think go usf life around though,nah think goe usf live around though



Preprocessing completed successfully!
Saved as: preprocessed_sms.csv
